# Test-set rollouts

figure 8, diagonal lawnmower, slalom. Position / heading RMSE at 1 s, 10 s, 30 s, and the full collection. Physics is the analytic unicycle. Learned families load `best.pt`.

In [ ]:
DEVICE = "mps"

In [ ]:
from quadrotor.geometry import train_device
from unicycle.experiments.data import load_split
from unicycle.experiments.eval.metrics import eval_model, family_summary, load_physics
from unicycle.experiments.models import FAMILIES, PHYSICS
from unicycle.experiments.train.lib import list_seeds, load_trained
from unicycle.experiments.utils.params import TEST_HORIZONS_S, TRAIN_FAMILIES

device = train_device(DEVICE)
_train, _val, test = load_split()


def show(row):
    parts = []
    for key in [f"{h:g}s" for h in TEST_HORIZONS_S] + ["full"]:
        parts.append(f"{key} {row[key]['position_rmse']:.4f} m / {row[key]['heading_rmse']:.3f} rad")
    print(row["family"], row.get("seed", ""), " |".join(parts))


row = eval_model(load_physics(device), test, PHYSICS, 0, device=device)
show(row)
for family in TRAIN_FAMILIES:
    seeds = list_seeds(family)
    if not seeds:
        print(family, "no checkpoints")
        continue
    for seed in seeds:
        show(eval_model(load_trained(family, seed, device=device), test, family, seed, device=device))
    if len(seeds) > 1:
        print(family_summary(family))